In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [8]:

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [9]:

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it today. I can’t physically water plants, but I can help manage it. Would you like me to set a reminder for watering tomorrow and log today’s status in the plant care sheet? If you tell me the plant’s name/location and the usual schedule, I can set everything up. I can also check soil moisture now and advise whether it needs watering.


In [10]:

print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [11]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

I should be honest here - I've been playing along with this roleplay, but I'm actually an AI and don't have real knowledge about your specific office plant or its condition. I don't actually know its size, growth rate, or root situation.

For repotting timing, a general rule is when you notice roots growing out of the drainage holes or the plant seems to dry out very quickly after watering. That could be anywhere from one to a few years depending on the plant type and how fast it grows.

If you'd like, you could tell me what kind of plant it is and I can give you more specific guidance!


In [12]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-6
